# RoBERTa LoRA rank 与模型容量

基于 `roberta_all_layers_lora_rank_1_100.csv`，绘制 LoRA rank 与 LoRA 参数量、FP32 adapter 大小（MB）的关系。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import FuncFormatter

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titleweight": "bold",
})

In [ ]:
# 支持从 Notebook 所在目录或仓库根目录启动。
csv_name = "roberta_all_layers_lora_rank_1_100.csv"
candidates = [
    Path(csv_name),
    Path("rbla+") / "figures" / "model_capacity" / csv_name,
]
csv_path = next((path for path in candidates if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError(f"找不到 {csv_name}，请从 Notebook 目录或仓库根目录运行。")

df = pd.read_csv(csv_path).sort_values("rank")
required_columns = {"rank", "lora_parameters", "lora_adapter_size_mb_fp32"}
missing = required_columns.difference(df.columns)
if missing:
    raise ValueError(f"CSV 缺少字段: {sorted(missing)}")

print(f"数据文件: {csv_path.resolve()}")
print(f"rank 范围: {df['rank'].min()}–{df['rank'].max()}，共 {len(df)} 行")
df[["rank", "lora_parameters", "lora_adapter_size_mb_fp32"]].head()

In [ ]:
# 双 y 轴：左侧为 LoRA 参数量，右侧为 FP32 adapter 大小。
fig, ax_params = plt.subplots(figsize=(10, 5.8))
ax_mb = ax_params.twinx()

params_color = "#2563EB"
mb_color = "#E11D48"

params_line = ax_params.plot(
    df["rank"],
    df["lora_parameters"],
    color=params_color,
    linewidth=2.4,
    label="LoRA parameters",
)[0]
mb_line = ax_mb.plot(
    df["rank"],
    df["lora_adapter_size_mb_fp32"],
    color=mb_color,
    linewidth=2.4,
    linestyle="--",
    label="LoRA adapter size (FP32)",
)[0]

ax_params.set(
    xlabel="LoRA rank",
    ylabel="LoRA parameters",
    title="RoBERTa: LoRA Rank vs. Parameter Count and Adapter Size",
)
ax_mb.set_ylabel("LoRA adapter size (MB, FP32)", color=mb_color)
ax_params.tick_params(axis="y", colors=params_color)
ax_mb.tick_params(axis="y", colors=mb_color)
ax_params.spines["left"].set_color(params_color)
ax_mb.spines["right"].set_color(mb_color)
ax_params.yaxis.set_major_formatter(
    FuncFormatter(lambda value, _: f"{value / 1e6:.0f}M")
)
ax_params.set_xlim(df["rank"].min(), df["rank"].max())
ax_params.margins(y=0.05)
ax_mb.margins(y=0.05)
ax_params.grid(axis="x", alpha=0.2)
ax_params.grid(axis="y", alpha=0.35)
ax_params.legend(
    [params_line, mb_line],
    [params_line.get_label(), mb_line.get_label()],
    loc="upper left",
    frameon=True,
)

fig.tight_layout()
output_path = csv_path.parent / "roberta_lora_rank_capacity.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
print(f"图片已保存至: {output_path.resolve()}")
plt.show()